<a href="https://colab.research.google.com/github/NarendraRaoJami/J_V_Narendra_Rao_Internship_2026_College/blob/main/J_V_Narendra_Rao/DuQuant_BERT_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Device Specifications

| Specification         | Details                                      |
|-----------------------|----------------------------------------------|
| **Platform**          | Google Colaboratory                          |
| **Runtime Type**      | GPU                                          |
| **Accelerator**       | NVIDIA Tesla T4                              |
| **GPU Memory**        | 15 GB GDDR6                                  |
| **CUDA Cores**        | 2,560                                        |
| **Tensor Cores**      | 320 (2nd Gen)                                |
| **GPU Architecture**  | Turing (SM 7.5)                              |
| **CUDA Version**      | 12.2                                         |
| **CPU**               | Intel Xeon (2 vCPUs)                         |
| **System RAM**        | ~12.7 GB                                     |
| **Disk Space**        | ~78 GB                                       |
| **Python Version**    | 3.10.x                                       |
| **OS**                | Ubuntu 22.04 LTS (64-bit)                    |
| **Driver Version**    | 525.xx (NVIDIA)                              |

In [ ]:
!pip install -q transformers datasets evaluate accelerate pynvml scikit-learn scipy

In [2]:
import time, gc, warnings, threading
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate as ev
import torch.nn.functional as F
from functools import partial

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

MODELS = {
    'TinyBERT':   'huawei-noah/TinyBERT_General_4L_312D',
    'DistilBERT': 'distilbert-base-uncased',
    'AlBERT':     'albert-base-v2',
    'MobileBERT': 'google/mobilebert-uncased',
    'BERT-base':  'bert-base-uncased',
}

DATASETS = {
    'SST2': ('stanfordnlp/sst2',  None,   'validation',         'sentence',               'label'),
    'QNLI': ('nyu-mll/glue',      'qnli', 'validation',         ('question','sentence'),   'label'),
    'MNLI': ('nyu-mll/glue',      'mnli', 'validation_matched', ('premise','hypothesis'),  'label'),
    'QQP':  ('nyu-mll/glue',      'qqp',  'validation',         ('question1','question2'), 'label'),
    'RTE':  ('nyu-mll/glue',      'rte',  'validation',         ('sentence1','sentence2'), 'label'),
    # ---ADDED---
    'CoLA': ('nyu-mll/glue',      'cola', 'validation',         'sentence',               'label'),
    'MRPC': ('nyu-mll/glue',      'mrpc', 'validation',         ('sentence1','sentence2'), 'label'),
    'STSB': ('nyu-mll/glue',      'stsb', 'validation',         ('sentence1','sentence2'), 'label'),
    'WNLI': ('nyu-mll/glue',      'wnli', 'validation',         ('sentence1','sentence2'), 'label'),
}

BATCH_SIZE      = 32
MAX_SAMPLES     = 500
MAX_LENGTH      = 128
QUANT_BITS      = 4
POLL_INTERVAL_S = 0.005  # 5 ms polling interval for power sampler


Device: cuda


In [3]:
# ---------------------------------------------------------------------------
# pynvml calls the NVML C library directly (no process spawn overhead),
# allowing us to poll at ~5 ms intervals and time-average power across the
# entire forward pass instead of taking a single snapshot before it.
# ---------------------------------------------------------------------------
try:
    import pynvml
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print(f'pynvml ready | GPU: {pynvml.nvmlDeviceGetName(_nvml_handle)}')
except Exception as e:
    NVML_AVAILABLE = False
    print(f'pynvml unavailable ({e}) — energy will be reported as 0.0 mJ')


class PowerSampler:
    def __init__(self):
        self._samples = []
        self._running = False
        self._thread  = None

    def _poll(self):
        while self._running:
            if NVML_AVAILABLE:
                try:
                    # nvmlDeviceGetPowerUsage returns milliwatts directly (no conversion needed)
                    self._samples.append(pynvml.nvmlDeviceGetPowerUsage(_nvml_handle))
                except Exception:
                    pass
            time.sleep(POLL_INTERVAL_S)

    def start(self):
        self._samples = []
        self._running = True
        self._thread  = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        """Stop sampling and return mean power in mW (0.0 if unavailable)."""
        self._running = False
        self._thread.join(timeout=0.5)
        return float(np.mean(self._samples)) if self._samples else 0.0


pynvml ready | GPU: Tesla T4


In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from scipy.stats import pearsonr, spearmanr
import tempfile
import os


In [5]:
def get_model(hf_id, num_labels):
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=num_labels, ignore_mismatched_sizes=True
    ).to(DEVICE)
    return tok, model

def compute_rotation_matrix(weight):
    channel_norms = weight.abs().mean(dim=0)
    outlier_dims = torch.topk(channel_norms, k=max(1, len(channel_norms)//8)).indices

    d = weight.shape[1]
    R = torch.eye(d, dtype=weight.dtype)
    for idx in outlier_dims:
        i = idx.item()
        neighbor = (i + 1) % d
        theta = torch.atan2(weight[:, i].norm(), weight[:, neighbor].norm())
        c, s = torch.cos(theta), torch.sin(theta)
        R[i, i] = c;       R[i, neighbor] = -s
        R[neighbor, i] = s; R[neighbor, neighbor] = c
    return R

def zigzag_permutation(weight, n_blocks=4):
    """Step 2: Zigzag permutation to balance outliers across blocks."""
    d = weight.shape[1]
    block_size = d // n_blocks
    channel_norms = weight.abs().mean(dim=0)

    sorted_indices = torch.argsort(channel_norms, descending=True)
    perm = torch.zeros(d, dtype=torch.long)
    for i, idx in enumerate(sorted_indices):
        block = i % n_blocks
        pos_in_block = i // n_blocks
        perm[block * block_size + min(pos_in_block, block_size-1)] = idx
    return perm

def apply_duquant(layer):
    """Apply DuQuant: Rotation → Zigzag Permutation → Rotation → INT4 quantize."""
    with torch.no_grad():
        W = layer.weight.data.float()

        # Step 1: First rotation (redistribute outliers to adjacent channels)
        R1 = compute_rotation_matrix(W)
        W = W @ R1

        # Step 2: Zigzag permutation (balance outliers across blocks)
        perm = zigzag_permutation(W)
        W = W[:, perm]

        # Step 3: Second rotation (further smooth activation landscape)
        R2 = compute_rotation_matrix(W)
        W = W @ R2

        # Step 4: INT4 quantization (W4)
        max_val = W.abs().max()
        scale = max_val / 7.0  # INT4 range: -8 to 7
        W_q = torch.clamp(torch.round(W / scale), -8, 7) * scale

        layer.weight.data = W_q.to(layer.weight.dtype)
    return layer

def quantize_model(model):
    model.cpu()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            apply_duquant(module)
    return model

def memory_mb(model):
    with tempfile.NamedTemporaryFile(delete=False) as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / (1024 * 1024)

    os.remove(f.name)

    return round(size_mb, 2)


def tokenize(tok, texts):
    if isinstance(texts[0], tuple):
        return tok([t[0] for t in texts], [t[1] for t in texts],
                   truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
    return tok(texts, truncation=True, padding=True,
               max_length=MAX_LENGTH, return_tensors='pt')


def benchmark(model, tok, ds_cfg):
    path, config, split, text_col, label_col = ds_cfg
    ds = load_dataset(path, config, split=split) if config else load_dataset(path, split=split)
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

    t_start = time.perf_counter()
    model.eval()
    preds, labels, latencies, energies = [], [], [], []


    for i in range(0, len(ds), BATCH_SIZE):
        batch = ds[i:i+BATCH_SIZE]
        texts = list(zip(batch[text_col[0]], batch[text_col[1]])) if isinstance(text_col, tuple) else batch[text_col]
        first_param = next(model.parameters(), None)

        if first_param is not None:
            model_device = first_param.device
        else:
            model_device = torch.device("cpu")

        enc = {
            k: v.to(model_device)
            for k, v in tokenize(tok, texts).items()
        }

        sampler = PowerSampler()
        sampler.start()

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)

        if model_device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) * 1000

        avg_power_mw = sampler.stop()

        batch_size_actual = len(batch[label_col])
        energy_mj = (avg_power_mw * elapsed_ms * 1e-3) / batch_size_actual

        latencies.append(elapsed_ms / batch_size_actual)
        energies.append(energy_mj)
        if config == 'stsb':
            preds.extend(out.logits.squeeze(-1).cpu().tolist())
        else:
            preds.extend(out.logits.argmax(-1).cpu().tolist())
        labels.extend(batch[label_col])

    total_time = time.perf_counter() - t_start
    latency = np.mean(latencies)
    throughput = len(ds) / total_time
    energy = np.mean(energies)
    if config == "stsb":

        pearson = pearsonr(preds, labels)[0]
        spearman = spearmanr(preds, labels)[0]

        return {
            "Accuracy": np.nan,
            "Precision": np.nan,
            "Recall": np.nan,
            "F1": np.nan,
            "Pearson": round(pearson * 100, 2),
            "Spearman": round(spearman * 100, 2),
            "Latency": round(latency, 4),
            "Throughput": round(throughput, 1),
            "Energy": round(energy, 4)
        }

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="weighted", zero_division=0)
    rec = recall_score(labels, preds, average="weighted", zero_division=0)
    f1 = f1_score(labels, preds, average="weighted", zero_division=0)

    return {
        "Accuracy": round(acc * 100, 2),
        "Precision": round(prec * 100, 2),
        "Recall": round(rec * 100, 2),
        "F1": round(f1 * 100, 2),
        "Pearson": np.nan,
        "Spearman": np.nan,
        "Latency": round(latency, 4),
        "Throughput": round(throughput, 1),
        "Energy": round(energy, 4)
    }


In [ ]:
NUM_LABELS = {
    'SST2': 2,
    'QNLI': 2,
    'MNLI': 3,
    'QQP': 2,
    'RTE': 2,
    'CoLA': 2,
    'MRPC': 2,
    'STSB': 1,
    'WNLI': 2
}

results = []

for ds_name, ds_cfg in DATASETS.items():

    for model_name, hf_id in MODELS.items():

        print(f"{ds_name} | {model_name}")

        try:

            tok, fp32_model = get_model(
                hf_id,
                NUM_LABELS[ds_name]
            )

            duquant_model = quantize_model(fp32_model)

            mem = memory_mb(duquant_model)

            metrics = benchmark(
                duquant_model,
                tok,
                ds_cfg
            )

            results.append({
                "Dataset": ds_name,
                "Model": model_name,
                "Method": "DuQuant",
                "Bits": 4,
                "Memory (MB)": mem,
                "Latency (ms)": metrics["Latency"],
                "Throughput (sps)": metrics["Throughput"],
                "Energy (mJ)": metrics["Energy"],
                "Accuracy (%)": metrics["Accuracy"],
                "Precision (%)": metrics["Precision"],
                "Recall (%)": metrics["Recall"],
                "F1 (%)": metrics["F1"],
                "Pearson (%)": metrics["Pearson"],
                "Spearman (%)": metrics["Spearman"]
            })

            print("Done")

        except Exception as e:

            print("ERROR:", e)

        finally:

            try:
                del fp32_model
                del duquant_model
                del tok
            except:
                pass

            gc.collect()

In [7]:
df = pd.DataFrame(results).set_index(['Dataset', 'Model'])
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', '{:.4f}'.format)

for ds in df.index.get_level_values('Dataset').unique():
    print(f'\n========== {ds} ==========')
    print(df.loc[ds].to_string())
print()



========== SST2 ==========
             Method  Bits  Memory (MB)  Latency (ms)  Throughput (sps)  Energy (mJ)  Accuracy (%)  Precision (%)  Recall (%)  F1 (%)  Pearson (%)  Spearman (%)
Model                                                                                                                                                          
TinyBERT    DuQuant     4      54.7700       11.9740           79.1000     322.4313       46.2000        45.4100     46.2000 45.4000          NaN           NaN
DistilBERT  DuQuant     4     255.4500       56.0596           17.6000    1539.4605       51.0000        49.3100     51.0000 47.0600          NaN           NaN
AlBERT      DuQuant     4      44.5900      120.2918            8.3000    3373.7036       53.0000        28.0900     53.0000 36.7200          NaN           NaN
MobileBERT  DuQuant     4      94.2100       33.2526           29.6000     947.7543       47.0000        22.0900     47.0000 30.0500          NaN           NaN
BERT-base   